# Benchmarking pg_ivm Incremental Materialized Views (IMMV)

This notebook tests and benchmarks the use of the pg_ivm extension for incrementally maintained materialized views (IMMV) using the same schema as in db_initialization. It measures the time to insert data and update the IMMV via an AFTER trigger.

In [1]:
# 1. Import Required Libraries
import psycopg2
import psycopg2.extras
import numpy as np
import time

In [2]:
# 2. Set Up Database Connection and Table Names
DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_PREFIX = "ivmtest"
PATCH_TABLE = f"{TABLE_PREFIX}_patch"
PRED_PATCH_TABLE = f"{TABLE_PREFIX}_pred_patch"
IMMV_NAME = f"{TABLE_PREFIX}_patch_label_agg_immv"

In [3]:
# 3. Drop and Recreate Tables
with psycopg2.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {PRED_PATCH_TABLE} CASCADE;")
        cur.execute(f"DROP TABLE IF EXISTS {PATCH_TABLE} CASCADE;")
        cur.execute(f"DROP MATERIALIZED VIEW IF EXISTS {IMMV_NAME} CASCADE;")  # pg_ivm uses standard syntax for drop
    conn.commit()
print("Dropped old tables and IMMV if they existed.")

Dropped old tables and IMMV if they existed.


In [4]:
# 4. Create Tables
with psycopg2.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
        CREATE UNLOGGED TABLE {PATCH_TABLE}(
            id SERIAL NOT NULL,
            patch_uid integer NOT NULL,
            gt_label integer,
            event_ts timestamp with time zone NOT NULL DEFAULT now(),
            image_id integer,
            working_mag double precision,
            PRIMARY KEY(id)
        );
        """)
        cur.execute(f"""
        CREATE UNLOGGED TABLE {PRED_PATCH_TABLE}(
            id SERIAL NOT NULL,
            patch_uid bigint NOT NULL,
            embed_coords point,
            grid_cell_i int,
            grid_cell_j int,
            event_ts timestamp with time zone NOT NULL DEFAULT now(),
            pred_label integer,
            patch_coords point,
            PRIMARY KEY(id)
        );
        """)
        cur.execute(f"CREATE INDEX idx_{PRED_PATCH_TABLE}_grid_cells ON {PRED_PATCH_TABLE} (grid_cell_i, grid_cell_j);")
    conn.commit()
print("Created tables.")

Created tables.


In [5]:
# 5. Enable pg_ivm Extension
with psycopg2.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS pg_ivm;")
    conn.commit()
print("pg_ivm extension enabled.")

pg_ivm extension enabled.


In [6]:
# 6. Create Incrementally Maintained Materialized View (IMMV) with pg_ivm
view_def = f'''
    SELECT
        pp.pred_label,
        p.gt_label,
        pp.grid_cell_i,
        pp.grid_cell_j,
        COUNT(*) AS patch_count
    FROM {PRED_PATCH_TABLE} pp
    INNER JOIN {PATCH_TABLE} p ON pp.patch_uid = p.patch_uid
    GROUP BY pp.pred_label, p.gt_label, pp.grid_cell_i, pp.grid_cell_j
'''
with psycopg2.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """SELECT pgivm.create_immv(%s, %s);""",
            (IMMV_NAME, view_def)
        )
    conn.commit()
print("IMMV created using pgivm.create_immv with INNER JOIN.")

IMMV created using pgivm.create_immv with INNER JOIN.


In [7]:
# 7. Create AFTER INSERT Trigger for IMMV Maintenance
# pg_ivm automatically maintains the IMMV, but for explicit demonstration, we show the trigger creation (if needed for custom logic)
# For most use cases, pg_ivm handles this internally, so this is just illustrative
print("pg_ivm will maintain the IMMV automatically after inserts.")

pg_ivm will maintain the IMMV automatically after inserts.


In [8]:
# 8. Generate Synthetic Data for Insert
N = 1000

FINEST_LEVEL = 12  # 2^12 = 4096 cells per axis
GRID_SIZE = 2 ** FINEST_LEVEL  # 4096
NUM_LABELS = 5
rng = np.random.default_rng(42)

patch_uids   = np.arange(1, N + 1, dtype=int)
gt_labels    = rng.integers(0, NUM_LABELS, size=N).tolist()
image_ids    = rng.integers(1, 11, size=N).tolist()
working_mags = rng.choice([10.0, 20.0, 40.0], size=N).tolist()

embed_x = np.clip(rng.normal(loc=GRID_SIZE / 2, scale=GRID_SIZE * 0.15, size=N), 0.0, GRID_SIZE)
embed_y = np.clip(rng.normal(loc=GRID_SIZE / 2, scale=GRID_SIZE * 0.15, size=N), 0.0, GRID_SIZE)
patch_x = rng.uniform(0, 10000, size=N)
patch_y = rng.uniform(0, 10000, size=N)
pred_labels = rng.integers(0, NUM_LABELS, size=N).tolist()
grid_cell_i = (embed_x // 1).astype(int)
grid_cell_j = (embed_y // 1).astype(int)

patch_rows = [
    (int(patch_uids[i]), gt_labels[i], image_ids[i], working_mags[i])
    for i in range(N)
]
pred_patch_rows = [
    (
        int(patch_uids[i]),
        f"({embed_x[i]},{embed_y[i]})",
        int(grid_cell_i[i]),
        int(grid_cell_j[i]),
        pred_labels[i],
        f"({patch_x[i]},{patch_y[i]})",
    )
    for i in range(N)
]
print(f"Generated {N} synthetic patch and pred_patch rows.")

Generated 1000 synthetic patch and pred_patch rows.


In [9]:
# 9. Benchmark Bulk Insert and IMMV Update (multiprocessing COPY, db_initialization style)
import io
import multiprocessing as mp

def build_patch_buffer(patch_uids, gt_label):
    buf = io.StringIO()
    for uid in patch_uids:
        buf.write(f"{uid}\t{gt_label}\t\\N\t\\N\n")
    buf.seek(0)
    return buf

def build_pred_buffer(patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py):
    buf = io.StringIO()
    for i in range(len(patch_uids)):
        buf.write(
            f"{patch_uids[i]}\t({ex[i]},{ey[i]})\t{grid_i[i]}\t{grid_j[i]}\t{pred_labels[i]}\t({px[i]},{py[i]})\n"
        )
    buf.seek(0)
    return buf

def worker(worker_id, N, NUM_LABELS, GRID_SIZE, FINEST_LEVEL):
    import numpy as np
    import psycopg2
    local_rng = np.random.default_rng(2026 + worker_id)
    patch_uid_start = worker_id * N + 1
    patch_uid_end = patch_uid_start + N
    patch_uids = np.arange(patch_uid_start, patch_uid_end, dtype=int)
    gt_label = worker_id % NUM_LABELS
    image_ids = local_rng.integers(1, 11, size=N)
    working_mags = local_rng.choice([10.0, 20.0, 40.0], size=N)
    embed_x = np.clip(local_rng.normal(loc=GRID_SIZE / 2, scale=GRID_SIZE * 0.15, size=N), 0.0, GRID_SIZE)
    embed_y = np.clip(local_rng.normal(loc=GRID_SIZE / 2, scale=GRID_SIZE * 0.15, size=N), 0.0, GRID_SIZE)
    patch_x = local_rng.uniform(0, 10000, size=N)
    patch_y = local_rng.uniform(0, 10000, size=N)
    pred_labels = local_rng.integers(0, NUM_LABELS, size=N)
    grid_cell_i = (embed_x // 1).astype(int)
    grid_cell_j = (embed_y // 1).astype(int)
    patch_buf = build_patch_buffer(patch_uids, gt_label)
    pred_buf = build_pred_buffer(patch_uids, embed_x, embed_y, grid_cell_i, grid_cell_j, pred_labels, patch_x, patch_y)
    with psycopg2.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.copy_from(
                patch_buf,
                PATCH_TABLE,
                columns=("patch_uid", "gt_label", "image_id", "working_mag")
            )
            cur.copy_from(
                pred_buf,
                PRED_PATCH_TABLE,
                columns=("patch_uid", "embed_coords", "grid_cell_i", "grid_cell_j", "pred_label", "patch_coords")
            )
        conn.commit()

N_WORKERS = 4
N_PER_WORKER = len(patch_rows) // N_WORKERS
FINEST_LEVEL = 12
GRID_SIZE = 2 ** FINEST_LEVEL
NUM_LABELS = 5

t0 = time.perf_counter()
with mp.Pool(N_WORKERS) as pool:
    pool.starmap(worker, [(wid, N_PER_WORKER, NUM_LABELS, GRID_SIZE, FINEST_LEVEL) for wid in range(N_WORKERS)])
t1 = time.perf_counter()
print(f"Multiprocessing COPY bulk insert (including IMMV update) took {t1-t0:.3f} seconds.")

# Each worker generates and inserts its own chunk of data, similar to db_initialization style.

Multiprocessing COPY bulk insert (including IMMV update) took 0.116 seconds.


## Why does adding more workers make it slower?

- **IMMV (pg_ivm) is updated after every insert, and each worker triggers IMMV maintenance concurrently.**
- This causes heavy lock contention and serialization in PostgreSQL, especially with joins and aggregates in the IMMV.
- The IMMV update logic is not parallelized; concurrent inserts force the database to serialize updates to the view, causing workers to block each other.
- More workers = more contention, so total time increases instead of decreasing.

**What you can do:**
- For bulk loads, drop the IMMV, load data, then recreate or refresh the IMMV after all inserts.
- If you must keep the IMMV live, use fewer workers (even 1) for best throughput.
- Consider partitioning data or using separate tables for each worker, then merging after load.

Would you like a code cell to demonstrate the drop/recreate-IMMV pattern for fast bulk load?

In [10]:
# 9a. Investigate Multiprocessing COPY Slowness
import cProfile, pstats, io as sysio

def profile_worker(*args, **kwargs):
    pr = cProfile.Profile()
    pr.enable()
    worker(*args, **kwargs)
    pr.disable()
    s = sysio.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumtime')
    ps.print_stats(20)
    print(s.getvalue())

# Profile a single worker to see where time is spent
profile_worker(0, N_PER_WORKER, NUM_LABELS, GRID_SIZE, FINEST_LEVEL)

# Note: This will print a profile of the slowest parts of the worker logic. Look for slow DB operations, buffer building, or data generation.

         1003 function calls (998 primitive calls) in 0.042 seconds

   Ordered by: cumulative time
   List reduced from 160 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        4    0.000    0.000    0.074    0.018 /home/ray/anaconda3/lib/python3.13/selectors.py:435(select)
      5/4    0.000    0.000    0.036    0.009 /home/ray/anaconda3/lib/python3.13/asyncio/base_events.py:1960(_run_once)
        2    0.023    0.012    0.023    0.012 {method 'copy_from' of 'psycopg2.extensions.cursor' objects}
        1    0.000    0.000    0.013    0.013 /home/ray/anaconda3/lib/python3.13/site-packages/decorator.py:232(fun)
        1    0.000    0.000    0.013    0.013 /home/ray/anaconda3/lib/python3.13/site-packages/IPython/core/history.py:100(only_when_enabled)
        1    0.000    0.000    0.007    0.007 /home/ray/anaconda3/lib/python3.13/site-packages/psycopg2/__init__.py:80(connect)
        1    0.006    0.006    0.006    0.006 {built

In [11]:
# 10. Verify IMMV Contents
with psycopg2.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {IMMV_NAME} LIMIT 10;")
        rows = cur.fetchall()
        for row in rows:
            print(row)
print("Verified IMMV contents (first 10 rows shown above).")

(3, 1, 2612, 2742, 1, 1)
(4, 1, 2291, 1418, 1, 1)
(1, 1, 3203, 2875, 1, 1)
(0, 1, 840, 2835, 1, 1)
(1, 1, 453, 2395, 1, 1)
(4, 1, 3152, 1876, 1, 1)
(2, 1, 943, 3360, 1, 1)
(1, 1, 2701, 2261, 1, 1)
(4, 1, 2850, 2225, 1, 1)
(3, 1, 2501, 184, 1, 1)
Verified IMMV contents (first 10 rows shown above).
